In [ ]:
#| default_exp preprocessing.create_tf_dataset

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
import sys
from pathlib import Path
import yaml
from platform import system
import albumentations as A

In [ ]:
#| export
import tensorflow as tf
import tensorflow_addons as tfa
from functools import partial
from typing import Union

In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))

In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))

In [ ]:
#| export
from cv_tools.imports import *
from dotenv import load_dotenv
from cv_tools.core import *

In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/homes/hasan/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| export
if system() != 'Windows':
    CURRENT_NB=Path('/home/ai_sintercra/homes/hasan/projects/git_data/new_test/nbs')
    os.chdir(CURRENT_NB)

In [ ]:
#| export
def get_config(config_path:str):
    'Get config from yaml file'
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

In [ ]:
#| export
def  get_common_fns(
        config:dict,
        training:bool=True
        ):
    'From config image and mask path get common files'
    if training:
        mask_names = set(get_name_(Path(config['dataset']['trn_msk_path']).ls()))
        images_names = set(get_name_(Path(config['dataset']['trn_im_path']).ls()))
    else:
        mask_names = set(get_name_(Path(config['dataset']['val_msk_path']).ls()))
        images_names = set(get_name_(Path(config['dataset']['val_im_path']).ls()))

    print(f' Number of mask files = {len(mask_names)}')
    print(f' Number of image files = {len(images_names)}')
    common_files = list(mask_names.intersection(images_names))
    print(f' Number of common files = {len(common_files)}')
    if training:
        ims = [Path(config['dataset']['trn_im_path'],i).as_posix() for i in common_files]
        msks = [Path(config['dataset']['trn_msk_path'], i).as_posix() for i in common_files]
    else:
        ims = [Path(config['dataset']['val_im_path'], i).as_posix() for i in common_files]
        msks = [Path(config['dataset']['val_msk_path'], i).as_posix() for i in common_files]
    return ims, msks

In [ ]:
#| export
def mixup_segmentation(
        images, 
        masks, 
        ALPHA=0.2):
    ' Mix up Augmentation following beta dist'
    if ALPHA <= 0:
        return images, masks

    batch_size = tf.shape(images)[0]
    weights = tf.random.uniform((batch_size, 1, 1, 1), minval=0, maxval=1)
    mix_weights = tf.maximum(weights, 1 - weights)  # Ensure symmetry

    # Randomly permute the batch to mix
    perm = tf.random.shuffle(tf.range(batch_size))
    perm_images = tf.gather(images, perm, axis=0)
    perm_masks = tf.gather(masks, perm, axis=0)

    mixed_images = mix_weights * images + (1 - mix_weights) * perm_images
    mixed_masks = mix_weights * masks + (1 - mix_weights) * perm_masks

    return mixed_images, mixed_masks

In [ ]:
#| export
def cutmix_segmentation(
        images, 
        masks, 
        PROBABILITY=.1):
    # Assumes images and masks are in the same batch and aligned
    if tf.random.uniform([]) > PROBABILITY:
        return images, masks

    batch_size, height, width, _ = images.shape
    mixing_images = images
    mixing_masks = masks



    # Randomly choose another element in the batch
    perm = tf.random.shuffle(tf.range(batch_size))
    perm_images = tf.gather(images, perm, axis=0)
    perm_masks = tf.gather(masks, perm, axis=0)

    # Randomly choose the mix ratio and the position to cut
    cut_rat = tf.sqrt(1 - tf.random.uniform([]) * 0.3)
    cut_h = tf.cast(height * cut_rat, tf.int32)
    cut_w = tf.cast(width * cut_rat, tf.int32)

    cx = tf.random.uniform([], int(width * 0.1), int(width * 0.9), tf.int32)
    cy = tf.random.uniform([], int(height * 0.1), int(height * 0.9), tf.int32)

    ymin = tf.clip_by_value(cy - cut_h // 2, 0, height)
    ymax = tf.clip_by_value(cy + cut_h // 2, 0, height)
    xmin = tf.clip_by_value(cx - cut_w // 2, 0, width)
    xmax = tf.clip_by_value(cx + cut_w // 2, 0, width)

    # Create masks that will be used to slice and mix the images
    mask1 = tf.concat([
        tf.ones((batch_size, ymin, width, 1)),
        tf.concat([
            tf.ones((batch_size, ymax - ymin, xmin, 1)),
            tf.zeros((batch_size, ymax - ymin, xmax - xmin, 1)),
            tf.ones((batch_size, ymax - ymin, width - xmax, 1))
        ], axis=2),
        tf.ones((batch_size, height - ymax, width, 1))
    ], axis=1)

    images = mask1 * images + (1 - mask1) * perm_images
    masks = mask1 * masks + (1 - mask1) * perm_masks

    return images, masks

In [ ]:
#| export
def normalize(
              image:Union[np.ndarray, tf.Tensor], 
              min=0):
    def _normalize(im):
        img = tf.cast(im, tf.float32)
        return img / 255.0

    if min == 0:
        return _normalize(image)
    else:
        return (_normalize(image) * 2.0) -1.0

In [ ]:
#| export
def read_normalize(
                    im_file:str,
                    config:dict,
                    ):
    im = tf.io.read_file(im_file)
    im = tf.image.decode_png(
        im, 
        channels=config['dataset']['image_channels'])


    # in albumentation normalizing is done
    im = normalize(im,min=config['dataset']['min'])
    im = tf.cast(im,
                 tf.float32)
    
                        
    return im

In [ ]:
#| export
def read_and_binarize_mask(
                    mask_file:str,
                    config:dict,
                    ):
    msk = tf.io.read_file(mask_file)
    msk = tf.image.decode_png(
        msk, 
        channels=config['dataset']['mask_channels'])
    mask = tf.cast(msk > 127, tf.float32)
    return mask

In [ ]:
#| export
def process_image_and_mask(
                          img_path,
                          msk_path, 
                          config):
    image = read_normalize(
                           im_file=img_path,
                           config=config)
    mask = read_and_binarize_mask(
                                mask_file=msk_path,
                                config=config)
    return image, mask

In [ ]:
#| export
def create_tf_ds(
        config:dict,
        training:bool=True,
        only_cutmix:bool=False,
        only_mixup:bool=False,
        both:bool=True
    ):
    """Create an optimized TensorFlow dataset for large image segmentation.
    
    Key optimizations:
    1. Uses parallel interleave for I/O bound operations
    2. Caches preprocessed images in memory/disk
    3. Optimizes augmentation pipeline
    4. Uses vectorized operations where possible
    5. Efficient prefetching and buffering
    
    Works efficiently on both CPU and multi-GPU setups by:
    - Using device-agnostic TF operations
    - Auto-tuning parallel operations based on available hardware
    - Optimizing memory usage and transfer
    """
    ims, msks = get_common_fns(config,training=training)
    
    # Create dataset with optimized parallel reading
    dataset = tf.data.Dataset.from_tensor_slices((ims, msks))
    
    # Use dynamic parallelism that adapts to available hardware
    num_parallel = tf.data.experimental.AUTOTUNE
    
    # Optimize parallel I/O while staying hardware-agnostic
    dataset = dataset.interleave(
        lambda x, y: tf.data.Dataset.from_tensors((x, y)),
        cycle_length=num_parallel,
        num_parallel_calls=num_parallel,
        deterministic=False
    )
    
    # Process images with hardware-optimized parallel calls
    dataset = dataset.map(
        partial(process_image_and_mask, config=config),
        num_parallel_calls=num_parallel
    )
    
    ## Cache preprocessed images - works on both CPU/GPU
    if tf.config.list_physical_devices('GPU'):
        # Use memory cache when GPUs are available
        dataset = dataset.cache()
    else:
        # Use disk cache for CPU-only to avoid memory pressure
        dataset = dataset.cache(f"/tmp/tf_cache_{hash(str(config))}")
    
    if training:
        # Optimize augmentation pipeline
        augmentation_fn = lambda x, y: (x, y)
        if only_cutmix:
            augmentation_fn = lambda x, y: cutmix_segmentation(x, y, PROBABILITY=0.5)
        elif only_mixup:
            augmentation_fn = lambda x, y: mixup_segmentation(x, y, ALPHA=0.2)
        elif both:
            def augmentation_fn(x, y):
                x, y = cutmix_segmentation(x, y, PROBABILITY=0.5)
                return mixup_segmentation(x, y, ALPHA=0.2)
                
        dataset = dataset.map(
            partial(augment_data, config=config),
            num_parallel_calls=num_parallel
        ).map(
            augmentation_fn,
            num_parallel_calls=num_parallel
        )
        
        # Adapt shuffle buffer to available memory
        buffer_size = min(50000, int(config['dataset']['buffer_size']))
        if not tf.config.list_physical_devices('GPU'):
            # Use smaller buffer on CPU
            buffer_size = min(buffer_size, 10000)
            
        dataset = dataset.shuffle(
            buffer_size=buffer_size,
            reshuffle_each_iteration=True
        )
    
    # Optimize batch processing for available hardware
    dataset = dataset.batch(
        config['training']['batch_size'],
        drop_remainder=True,
        num_parallel_calls=num_parallel
    )
    
    # Enable parallel prefetching scaled to hardware
    dataset = dataset.prefetch(buffer_size=num_parallel)
    
    if training:
        dataset = dataset.repeat()
        
    return dataset

In [ ]:
config_path = Path(Path.cwd().parent, 'configs')
config_fn = 'config_office_local.yaml'
if system() == 'Windows':
    CONFIG_PATH=Path(config_path, config_fn)

In [ ]:
config = get_config(CONFIG_PATH)

In [ ]:
#| export
def augmentation_f(image, mask, config):
    """Apply configurable augmentations to image and mask pairs.
    
    Args:
        image: Input image tensor
        mask: Input mask tensor 
        config: Dict containing augmentation parameters from config.yaml
    
    Returns:
        tuple: Augmented (image, mask) pair
    """
    # Get augmentation config from yaml
    aug_config = config['dataset']['augmentations']
    im_h = config['dataset']['image_height']
    im_w = config['dataset']['image_width']

    # Build augmentation pipeline dynamically from config
    transforms = []
    
    ## Add shift-scale-rotate if configured
    #if aug_config.get('p_shift_scale_rotate', 0) > 0:
        #transforms.append(
            #A.ShiftScaleRotate(
                #shift_limit=tuple(aug_config['shift_limit']),
                #scale_limit=tuple(aug_config['scale_limit']), 
                #rotate_limit=tuple(aug_config['rotate_limit']),
                #border_mode=aug_config['border_mode'],
                #p=aug_config['p_shift_scale_rotate']
            #)
        #)
    
    # Add basic geometric transforms
    if aug_config.get('p_horizontal_flip', 0) > 0:
        transforms.append(A.HorizontalFlip(p=aug_config['p_horizontal_flip']))
    if aug_config.get('p_vertical_flip', 0) > 0:
        transforms.append(A.VerticalFlip(p=aug_config['p_vertical_flip']))
    if aug_config.get('p_transpose', 0) > 0:
        transforms.append(A.Transpose(p=aug_config['p_transpose']))
        
    # Add color/intensity augmentations
    if aug_config.get('p_brightness_contrast', 0) > 0:
        transforms.append(A.RandomBrightnessContrast(p=aug_config['p_brightness_contrast']))
    if aug_config.get('p_color_jitter', 0) > 0:
        transforms.append(A.ColorJitter(brightness=0.7, p=aug_config['p_color_jitter']))
        
    # Add noise augmentations
    if aug_config.get('p_gauss_noise', 0) > 0:
        transforms.append(
            A.GaussNoise(
                var_limit=tuple(aug_config['gauss_noise_var_limit']), 
                p=aug_config['p_gauss_noise']
            )
        )
    if aug_config.get('p_gaussian_blur', 0) > 0:
        transforms.append(
            A.GaussianBlur(
                blur_limit=tuple(aug_config['gaussian_blur_limit']), 
                p=aug_config['p_gaussian_blur']
            )
        )
    if aug_config.get('p_multiplicative_noise', 0) > 0:
        transforms.append(
            A.MultiplicativeNoise(
                multiplier=tuple(aug_config['multiplicative_noise_multiplier']),
                per_channel=True,
                p=aug_config['p_multiplicative_noise'] 
            )
        )
        
    # Always add resize as final transform
    transforms.append(
        A.RandomResizedCrop(
            height=im_h,
            width=im_w,
            scale=(0.8, 1.0),
            ratio=(0.75, 1.33),
            p=0.5
        )
    )
    transforms.append(A.Resize(
        height=im_h, 
        width=im_w, 
        p=1))

    # Create and apply augmentation pipeline
    aug = A.Compose(transforms)
    augmented = aug(image=image, mask=mask)
    
    return augmented['image'], augmented['mask']

In [ ]:
#| export
def augment_data(
                image, 
                mask, 
                config):
    aug_config = config['dataset']['augmentations']
    data_config = config['dataset']
    image_shape = (
                    data_config['image_height'], 
                    data_config['image_width'], 
                    data_config['image_channels']
                 )

    mask_shape = (
                    data_config['image_height'], 
                    data_config['image_width'], 
                    data_config['mask_channels']
                 )

    aug_func = partial(augmentation_f, config=config)

    aug_img, aug_mask = tf.numpy_function(
        func=aug_func,
        inp=[
            image,
            mask
        ],
        Tout=(tf.float32, tf.float32)
    )

    aug_img.set_shape(image_shape)
    aug_mask.set_shape(mask_shape)

    print(f'Augmented image shape = {aug_img.shape}') 
    print(f'Augmented image shape = {aug_mask.shape}') 

    aug_mask = tf.cast(aug_mask > 0.5, tf.float32)

    return aug_img, aug_mask

In [ ]:
ds_tf = create_tf_ds(
    config, 
    training=True, 
    only_cutmix=False, 
    only_mixup=False, 
    both=False)

 Number of mask files = 17894
 Number of image files = 17894
 Number of common files = 17894
Augmented image shape = (1152, 1632, 1)
Augmented image shape = (1152, 1632, 1)


In [ ]:
#| export
def get_data(config):
    trn_ds = create_tf_ds(
        config, 
        training=True,
        only_cutmix=False, 
        only_mixup=False, 
        both=False)
    val_ds = create_tf_ds(
        config, 
        training=False,
        only_cutmix=False, 
        only_mixup=False, 
        both=False)
    return trn_ds, val_ds

In [ ]:
trn_ds, val_ds = get_data(config)

 Number of mask files = 17894
 Number of image files = 17894
 Number of common files = 17894
Augmented image shape = (1152, 1632, 1)
Augmented image shape = (1152, 1632, 1)
 Number of mask files = 2687
 Number of image files = 2686
 Number of common files = 2686


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('37_preprocessing.create_tf_dataset.ipynb')

ValueError: '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\new_test\\nbs\\13_noise_lighting.ipynb' is not in the subpath of '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\nbs' OR one path is relative and the other is absolute.